# prior
taking previous work on QUBO

In [4]:
import numpy as np
from itertools import combinations

from numba import njit #optimization
import matplotlib.pyplot as plt

import time

#dwave
#import tabu
#import neal


In [ ]:
def estimate_temperature_range(Q, n_samples=100, hot_accept_prob=0.5, cold_accept_prob=1e-3):
    n = Q.shape[0]
    all_dEs = []

    for _ in range(n_samples):
        x = np.random.choice(np.array([0, 1]), size=n).astype(np.float64)
        for i in range(n):
            dE = delta_energy(x, Q, i)
            all_dEs.append(abs(dE))

    all_dEs = np.array(all_dEs)
    all_dEs = all_dEs[all_dEs > 0]   # ignore zero-moves, they don't inform temperature scale

    dE_typical = np.percentile(all_dEs, 50)   # median dE, for T_start
    dE_small   = np.percentile(all_dEs, 5)    # a small dE, for T_end

    T_start = -dE_typical / np.log(hot_accept_prob)
    T_end   = -dE_small / np.log(cold_accept_prob)

    return T_start, T_end, all_dEs


def estimate_runtime_converge_fast(Q, T_start, T_end, alpha, window, tol, n_reads,
                                     n_probe_early=10, n_probe_late=10,
                                      max_sweeps=5000):
    n = Q.shape[0]
    x = np.random.choice([0, 1], size=n).astype(np.float64)
    E = energy_QUBO(x, Q)

    _ = sweep_once_QUBO(x, Q, 100.0, E)  # JIT warm-up

    start = time.time()
    for _ in range(1000):
        E = sweep_once_QUBO(x, Q, 100.0, E)
    time_per_sweep = (time.time() - start) / 1000

    _ = sweep_until_converged(x, Q, T_start, E, window, tol, max_sweeps)  # JIT warm-up

    num_temp_steps_total = int(np.log(T_end / T_start) / np.log(alpha))

    # --- probe early steps (run the real schedule from the start) ---
    x_probe = np.random.choice([0, 1], size=n).astype(np.float64)
    E_probe = energy_QUBO(x_probe, Q)
    T = T_start
    early_sweeps = []

    n_early = min(n_probe_early, num_temp_steps_total)
    for _ in range(n_early):
        E_probe, s_done = sweep_until_converged(x_probe, Q, T, E_probe, window, tol, max_sweeps)
        early_sweeps.append(s_done)
        T *= alpha

    # --- jump ahead to probe late steps ---
    n_late = max(0, min(n_probe_late, num_temp_steps_total - n_early))   # <-- CHANGED: clamp to >= 0

    late_sweeps = []
    if n_late > 0:                                                        # <-- CHANGED: guard the block
        T_late_start = T_end / (alpha ** n_late)
        T = max(T, T_late_start)

        for _ in range(n_late):
            E_probe, s_done = sweep_until_converged(x_probe, Q, T, E_probe, window, tol, max_sweeps)
            late_sweeps.append(s_done)
            T *= alpha

    # --- combine estimate ---
    all_probe_sweeps = early_sweeps + late_sweeps
    avg_sweeps_per_step = np.mean(all_probe_sweeps)
    est_total_sweeps = avg_sweeps_per_step * num_temp_steps_total
    est_time_per_read = time_per_sweep * est_total_sweeps

    print(f"time per sweep:             {time_per_sweep*1000:.4f} ms")
    print(f"estimated time per anneal:  {est_time_per_read:.2f} s")

    time_total = est_time_per_read * n_reads
    if n_reads > 1:
        print(f"estimated time for {n_reads} reads: {time_total/60:.2f} min ({time_total:.2f} s)")

    return time_per_sweep, est_time_per_read, time_total, est_total_sweeps

In [ ]:
@njit
def energy_QUBO(x, Q):
    return x @ Q @ x

   
def sim_annealing_QUBO_converge(Q, alpha, window=50, tol=1e-3, max_sweeps=5000,verbose=False):
    
    T_start, T_end, _ = estimate_temperature_range(Q,
        n_samples=20, hot_accept_prob=0.5, cold_accept_prob=1e-3)
    
    if verbose==True:
        print(f"Starting temp: {T_start:.2f}\nEnding temp: {T_end:.2f}")
        estimate_runtime_converge_fast(Q, T_start, T_end, alpha, window, tol,
            n_reads,n_probe_early=10, n_probe_late=1, max_sweeps=5000)
            
    n = Q.shape[0]
    x = np.random.choice(np.array([0, 1]), size=n).astype(np.float64)

    E = energy_QUBO(x, Q)

    temps = []
    energies = []
    sweeps_used = []


    T = T_start
    
    while T > T_end:
        E, s_done = sweep_until_converged(x, Q, T, E, window, tol, max_sweeps)

        #test
        #E_true = energy_QUBO(x,Q)
        #print(T, E-E_true)

        temps.append(T)
        energies.append(E)
        sweeps_used.append(s_done)
        
        T *= alpha
    
    return np.array(temps), np.array(energies), x.copy()

@njit
def delta_energy(x, Q, i):
    s = 0.0
    N = len(x)

    for j in range(N):
        if j != i:
            s += Q[i, j] * x[j]

    if x[i] == 0:
        # 0 -> 1
        return Q[i, i] + 2.0 * s
    else:
        # 1 -> 0
        return -(Q[i, i] + 2.0 * s)

@njit
def sweep_until_converged(x, Q, T, E, window=50, tol=1e-3, max_sweeps=5000):
    recent_energies = np.zeros(window)
    sweeps_done = 0
    for s in range(max_sweeps):
        E = sweep_once_QUBO(x, Q, T, E)
        recent_energies[s % window] = E
        sweeps_done += 1
        if s >= window:
            spread = recent_energies.max() - recent_energies.min()
            if spread < tol * abs(E):
                break
    return E, sweeps_done